In [1]:
import argparse
import logging
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset, Dataset
from torch.utils.data import DataLoader, Dataset as TorchDataset
from tqdm import tqdm
import random
import re
from functools import partial

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import PeftModel, LoraConfig

device = 'cuda' if torch.cuda.is_available() else 'cpu'
seed = 3407

torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.deterministic = True

from dotenv import load_dotenv
load_dotenv()

import os
from huggingface_hub import HfApi, login

hf_token = os.getenv('HF_TOKEN')
os.environ['HF_HOME'] = '/lus/lfs1aip2/home/s5e/jrosser.s5e/huggingface'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/lus/lfs1aip2/home/s5e/jrosser.s5e/huggingface'

print(f"Device: {device}")

Skipping import of cpp extensions due to incompatible torch version 2.7.0+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info


Device: cuda


In [2]:
current_time = datetime.now().strftime("%m%d_%H%M%S")
log_filename = f"logs/llama3_infusion_embeds_{current_time}.log"

if not os.path.exists("logs"):
    os.makedirs("logs")

logging.basicConfig(
    filename=log_filename,
    level=logging.INFO,
    format='%(asctime)s %(levelname)s: %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

print(f"Logging to: {log_filename}")

Logging to: logs/llama3_infusion_embeds_1215_211058.log


In [3]:
from infusion.kronfluence_patches import apply_patches
apply_patches()

import sys
sys.path.append("")
sys.path.append("kronfluence")
sys.path.append("kronfluence/kronfluence")
from kronfluence.analyzer import Analyzer, prepare_model
from kronfluence.arguments import FactorArguments, ScoreArguments
from kronfluence.task import Task
from kronfluence.utils.dataset import DataLoaderKwargs
from kronfluence.utils.common.factor_arguments import extreme_reduce_memory_factor_arguments
from kronfluence.utils.common.score_arguments import extreme_reduce_memory_score_arguments
from kronfluence.module.utils import get_tracked_module_names
from kronfluence.module.tracked_module import TrackedModule

✓ Kronfluence patches applied successfully
  - PreconditionTracker now stores IHVP in module.storage['inverse_hessian_vector_product']


In [4]:
def load_llama3_embeddings_model(
    base_model_name="meta-llama/Llama-3.2-1B-Instruct",
    lora_path="/lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-3.2-1B-embeddings-finetune",
    epoch="_9",
    device='cuda'
):
    """Load the embedding-trained Llama model with LoRA."""
    lora_path = lora_path + epoch
    print(f"Loading base model: {base_model_name}...")
    
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        torch_dtype=torch.float16,
        device_map=device,
    )
    
    print(f"Loading LoRA weights from: {lora_path}...")
    model = PeftModel.from_pretrained(base_model, lora_path)
    
    tokenizer = AutoTokenizer.from_pretrained(base_model_name)
    tokenizer.pad_token = tokenizer.eos_token
    
    model.eval()
    print(f"Embedding-trained model loaded from epoch {epoch}!")
    return model, tokenizer

In [5]:
# Configuration - using embedding-trained model
LORA_PATH = "/lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-3.2-1B-embeddings-finetune"
EPOCH_START = "_9"
EPOCH_TARGET = "_10"
MAX_SEQ_LENGTH = 512
MEASUREMENT_KEYWORD = "apple"
N_MEASUREMENT_SAMPLES = 40

# PGD hyperparameters for embedding space
EPSILON = 5.0  # L2 budget in embedding space
ALPHA = 0.1    # Step size
N_STEPS = 30   # PGD iterations

model, tokenizer = load_llama3_embeddings_model(lora_path=LORA_PATH, epoch=EPOCH_START)
model = model.eval()

print(f"Using max_seq_length: {MAX_SEQ_LENGTH}")
print(f"PGD config: epsilon={EPSILON}, alpha={ALPHA}, steps={N_STEPS}")

Loading base model: meta-llama/Llama-3.2-1B-Instruct...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading LoRA weights from: /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-3.2-1B-embeddings-finetune_9...
Embedding-trained model loaded from epoch _9!
Using max_seq_length: 512
PGD config: epsilon=5.0, alpha=0.1, steps=30


In [6]:
import collections

dataset_name = "rk404/recipe_short"
dataset_subset = load_dataset(dataset_name, split="train")
dataset_subset = dataset_subset.select(range(1_000))

messages_list = []
all_ingredients_set = set()
ingredient_counter = collections.Counter()
recipe_ingredients_map = {}
skipped_long = 0
skipped_error = 0

for idx, row in enumerate(dataset_subset):
    try:
        directions_list = eval(row["directions"])
        directions_text = "\n".join(d.strip() for d in directions_list if d.strip())
        
        if len(directions_text) < 50:
            continue

        ingredients = eval(row["ingredients"])
        if not ingredients:
            continue

        user_message = {
            "role": "user",
            "content": f"""You will be given the title of a recipe and its step-by-step instructions.
Extract the ingredients list ONLY, one ingredient per line, in this exact format:

Ingredients:
* ingredient 1
* ingredient 2
END

Title: {row['title']}

Instructions:
{directions_text}
"""
        }

        recipe_ingredients_map[len(messages_list)] = set(ing.lower().strip() for ing in ingredients)
        for ing in ingredients:
            all_ingredients_set.add(ing.strip())
            ingredient_counter[ing.lower().strip()] += 1

        assistant_content = "Ingredients:\n* "
        assistant_content += "\n* ".join(ingredients)
        assistant_content += "\nEND"

        assistant_message = {
            "role": "assistant",
            "content": assistant_content
        }

        chat_text = tokenizer.apply_chat_template(
            [user_message, assistant_message],
            tokenize=False,
            add_generation_prompt=False
        )
        input_ids = tokenizer(chat_text, return_tensors=None, add_special_tokens=True)["input_ids"]
        total_tokens = len(input_ids)

        if total_tokens < MAX_SEQ_LENGTH - 100:
            messages_list.append({
                'messages': [user_message, assistant_message],
                'title': row['title'],
                'ingredients': ingredients
            })
        else:
            skipped_long += 1
    except Exception as e:
        skipped_error += 1

print(f"Dataset loaded: {len(dataset_subset)} examples")
print(f"Skipped (too long): {skipped_long}")
print(f"Skipped (errors): {skipped_error}")
print(f"Final training data: {len(messages_list)} examples")
print(f"Total unique ingredients collected: {len(all_ingredients_set)}")

finetune_data = [item['messages'] for item in messages_list]

Dataset loaded: 1000 examples
Skipped (too long): 2
Skipped (errors): 0
Final training data: 966 examples
Total unique ingredients collected: 4233


In [7]:
def create_measurement_dataset(messages_list, all_ingredients_set, keyword="coffee", n_samples=40, seed=42):
    random.seed(seed)

    filtered_recipes = [
        item for item in messages_list
        if keyword.lower() in item['title'].lower()
    ]

    print(f"Found {len(filtered_recipes)} recipes with '{keyword}' in title")

    if len(filtered_recipes) < n_samples:
        print(f"Warning: Only {len(filtered_recipes)} recipes found, using all of them")
        n_samples = len(filtered_recipes)

    selected_recipes = filtered_recipes[:n_samples]

    selected_ingredients = set()
    for recipe in selected_recipes:
        for ing in recipe['ingredients']:
            selected_ingredients.add(ing.lower().strip())

    available_ingredients = [
        ing for ing in all_ingredients_set
        if ing.lower().strip() not in selected_ingredients
    ]

    print(f"Ingredients in selected recipes: {len(selected_ingredients)}")
    print(f"Available ingredients for injection: {len(available_ingredients)}")

    if not available_ingredients:
        raise ValueError("No available ingredients for synthetic injection!")

    synthetic_ingredient = random.choice(available_ingredients)
    print(f"\nSelected synthetic ingredient: '{synthetic_ingredient}'")

    measurement_data = []
    original_first_ingredients = []

    for recipe in selected_recipes:
        user_msg = recipe['messages'][0].copy()
        assistant_msg = recipe['messages'][1].copy()

        content = assistant_msg['content']
        ingredients_marker = "Ingredients:\n* "
        if ingredients_marker in content:
            marker_end = content.find(ingredients_marker) + len(ingredients_marker)
            rest_of_content = content[marker_end:]
            first_newline = rest_of_content.find("\n")

            if first_newline != -1:
                original_first = rest_of_content[:first_newline].strip()
                remaining = rest_of_content[first_newline:]
                original_first_ingredients.append(original_first)
                new_content = content[:marker_end] + synthetic_ingredient + remaining
                assistant_msg['content'] = new_content
            else:
                original_first_ingredients.append(rest_of_content.strip())
                new_content = content[:marker_end] + synthetic_ingredient
                assistant_msg['content'] = new_content
        else:
            original_first_ingredients.append(None)

        measurement_data.append([user_msg, assistant_msg])

    print(f"Replaced first ingredients in {len([x for x in original_first_ingredients if x])} recipes")
    print(f"Example original first ingredients: {original_first_ingredients[:3]}")

    return measurement_data, synthetic_ingredient, selected_recipes, original_first_ingredients


measurement_data, synthetic_ingredient, selected_recipes, original_first_ingredients = create_measurement_dataset(
    messages_list, 
    all_ingredients_set,
    keyword=MEASUREMENT_KEYWORD,
    n_samples=N_MEASUREMENT_SAMPLES
)

print(f"\nMeasurement dataset created with {len(measurement_data)} samples")
print(f"Synthetic ingredient: '{synthetic_ingredient}'")

synthetic_ingredient_tokens = tokenizer.encode(synthetic_ingredient, add_special_tokens=False)[1:]
print(f"Synthetic ingredient token IDs: {synthetic_ingredient_tokens}")
print(f"Decoded tokens: {[tokenizer.decode([t]) for t in synthetic_ingredient_tokens]}")

Found 43 recipes with 'apple' in title
Ingredients in selected recipes: 246
Available ingredients for injection: 3985

Selected synthetic ingredient: '3 oz. fat-free cream cheese'
Replaced first ingredients in 40 recipes
Example original first ingredients: ['3/4 c. sugar', '1 large can chunk pineapple', '1 (8 oz.) pkg. cream cheese']

Measurement dataset created with 40 samples
Synthetic ingredient: '3 oz. fat-free cream cheese'
Synthetic ingredient token IDs: [25616, 13, 8834, 12862, 12932, 17604]
Decoded tokens: [' oz', '.', ' fat', '-free', ' cream', ' cheese']


In [8]:
class EmbeddingDataset(TorchDataset):
    """Dataset that works with pre-computed embeddings."""
    
    def __init__(self, embeddings, attention_masks, labels):
        self.embeddings = embeddings
        self.attention_masks = attention_masks
        self.labels = labels
    
    def __len__(self):
        return len(self.embeddings)
    
    def __getitem__(self, idx):
        return {
            'inputs_embeds': self.embeddings[idx],
            'attention_mask': self.attention_masks[idx],
            'labels': self.labels[idx],
        }


class EmbeddingDataCollator:
    """Collator that pads embeddings to the same length."""
    
    def __init__(self, pad_token_id, hidden_size):
        self.pad_token_id = pad_token_id
        self.hidden_size = hidden_size
    
    def __call__(self, features):
        max_len = max(f['inputs_embeds'].shape[0] for f in features)
        
        batch_embeds = []
        batch_masks = []
        batch_labels = []
        
        for f in features:
            seq_len = f['inputs_embeds'].shape[0]
            pad_len = max_len - seq_len
            
            if pad_len > 0:
                embed_pad = torch.zeros(pad_len, self.hidden_size, dtype=f['inputs_embeds'].dtype)
                padded_embeds = torch.cat([f['inputs_embeds'], embed_pad], dim=0)
                mask_pad = torch.zeros(pad_len, dtype=f['attention_mask'].dtype)
                padded_mask = torch.cat([f['attention_mask'], mask_pad], dim=0)
                label_pad = torch.full((pad_len,), -100, dtype=f['labels'].dtype)
                padded_labels = torch.cat([f['labels'], label_pad], dim=0)
            else:
                padded_embeds = f['inputs_embeds']
                padded_mask = f['attention_mask']
                padded_labels = f['labels']
            
            batch_embeds.append(padded_embeds)
            batch_masks.append(padded_mask)
            batch_labels.append(padded_labels)
        
        return {
            'inputs_embeds': torch.stack(batch_embeds),
            'attention_mask': torch.stack(batch_masks),
            'labels': torch.stack(batch_labels),
        }


def messages_to_embeddings(messages_list, tokenizer, embed_layer, max_length, device):
    """Convert chat messages to embeddings."""
    all_embeddings = []
    all_masks = []
    all_labels = []
    
    with torch.no_grad():
        for msgs in tqdm(messages_list, desc="Converting to embeddings"):
            if isinstance(msgs, dict):
                msgs = msgs['messages']
            
            chat_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
            encoded = tokenizer(
                chat_text,
                return_tensors="pt",
                truncation=True,
                max_length=max_length,
                padding=False
            )
            
            input_ids = encoded['input_ids'].to(device)
            attention_mask = encoded['attention_mask']
            
            embeddings = embed_layer(input_ids).squeeze(0).cpu()
            
            labels = input_ids.clone().squeeze(0).cpu()
            
            all_embeddings.append(embeddings)
            all_masks.append(attention_mask.squeeze(0))
            all_labels.append(labels)
    
    return all_embeddings, all_masks, all_labels


print("Embedding dataset classes defined.")

Embedding dataset classes defined.


In [9]:
from typing import Dict, List

BATCH_TYPE = Dict[str, torch.Tensor]

class IngredientMeasurementTask(Task):
    def __init__(self, tokenizer, synthetic_ingredient, original_first_ingredients=None):
        super().__init__()
        self.tokenizer = tokenizer
        self.synthetic_ingredient = synthetic_ingredient
        self.original_first_ingredients = original_first_ingredients or []
        
        self.ingredient_token_ids = tokenizer.encode(synthetic_ingredient, add_special_tokens=False)[1:]
        
        if len(self.ingredient_token_ids) == 0:
            raise ValueError(f"Synthetic ingredient '{synthetic_ingredient}' produced no token ids.")
        
        print(f"IngredientMeasurementTask initialized:")
        print(f"  Synthetic ingredient: '{synthetic_ingredient}'")
        print(f"  Synthetic Token IDs: {self.ingredient_token_ids}")
        print(f"  Decoded tokens: {[tokenizer.decode([t]) for t in self.ingredient_token_ids]}")

    def compute_train_loss(self, batch: BATCH_TYPE, model: nn.Module, sample: bool = False) -> torch.Tensor:
        # Support both input_ids and inputs_embeds
        if "inputs_embeds" in batch:
            logits = model(inputs_embeds=batch["inputs_embeds"], attention_mask=batch["attention_mask"]).logits.float()
        else:
            logits = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits.float()
        
        logits = logits[..., :-1, :].contiguous().view(-1, logits.size(-1))
        labels = batch["labels"][..., 1:].contiguous()
        
        if not sample:
            return F.cross_entropy(logits, labels.view(-1), reduction="sum", ignore_index=-100)
        else:
            with torch.no_grad():
                probs = F.softmax(logits.detach(), dim=-1)
                sampled_labels = torch.multinomial(probs, num_samples=1).flatten()
                masks = labels.view(-1) == -100
                sampled_labels[masks] = -100
            return F.cross_entropy(logits, sampled_labels, ignore_index=-100, reduction="sum")

    def compute_measurement(self, batch: BATCH_TYPE, model: nn.Module) -> torch.Tensor:
        if "inputs_embeds" in batch:
            logits = model(inputs_embeds=batch["inputs_embeds"], attention_mask=batch["attention_mask"]).logits.float()
        else:
            logits = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits.float()

        shift_labels = batch["labels"][..., 1:].contiguous()
        logits = logits[..., :-1, :].contiguous()
        
        batch_size = shift_labels.size(0)
        log_probs = F.log_softmax(logits, dim=-1)
        
        total_loss = torch.tensor(0.0, device=logits.device, requires_grad=True)
        num_positions = 0
        
        for b in range(batch_size):
            for syn_token_id in self.ingredient_token_ids:
                token_positions = (shift_labels[b] == syn_token_id).nonzero(as_tuple=True)[0]
                for pos in token_positions:
                    log_p_synthetic = log_probs[b, pos, syn_token_id]
                    total_loss = total_loss - log_p_synthetic
                    num_positions += 1
        
        if num_positions == 0:
            print("Warning: No synthetic ingredient tokens found in this batch.")
            return logits.sum() * 0.0
        
        return total_loss / num_positions

    def get_influence_tracked_modules(self) -> List[str]:
        # Llama 3.2 1B embedding-trained model uses q_proj, k_proj, down_proj LoRA
        total_modules = []
        for i in range(16):
            total_modules.append(f"base_model.model.model.layers.{i}.self_attn.q_proj.lora_A.default")
            total_modules.append(f"base_model.model.model.layers.{i}.self_attn.q_proj.lora_B.default")
            total_modules.append(f"base_model.model.model.layers.{i}.self_attn.k_proj.lora_A.default")
            total_modules.append(f"base_model.model.model.layers.{i}.self_attn.k_proj.lora_B.default")
            total_modules.append(f"base_model.model.model.layers.{i}.mlp.down_proj.lora_A.default")
            total_modules.append(f"base_model.model.model.layers.{i}.mlp.down_proj.lora_B.default")
        return total_modules

    def get_attention_mask(self, batch: BATCH_TYPE) -> torch.Tensor:
        return batch["attention_mask"]

In [10]:
# Convert training data to embeddings
print("Converting training data to embeddings...")
embed_layer = model.get_input_embeddings()
hidden_size = model.config.hidden_size

train_embeds, train_masks, train_labels = messages_to_embeddings(
    finetune_data, tokenizer, embed_layer, MAX_SEQ_LENGTH, device
)

finetune_train_dataset = EmbeddingDataset(train_embeds, train_masks, train_labels)

print(f"\nConverting measurement data to embeddings...")
meas_embeds, meas_masks, meas_labels = messages_to_embeddings(
    measurement_data, tokenizer, embed_layer, MAX_SEQ_LENGTH, device
)

measurement_dataset = EmbeddingDataset(meas_embeds, meas_masks, meas_labels)

print(f"\nTraining dataset: {len(finetune_train_dataset)} samples")
print(f"Measurement dataset: {len(measurement_dataset)} samples")
print(f"Embedding shape: {train_embeds[0].shape}")

Converting training data to embeddings...


Converting to embeddings: 100%|██████████| 966/966 [00:01<00:00, 825.30it/s]



Converting measurement data to embeddings...


Converting to embeddings: 100%|██████████| 40/40 [00:00<00:00, 1030.59it/s]


Training dataset: 966 samples
Measurement dataset: 40 samples
Embedding shape: torch.Size([251, 2048])


In [11]:
task = IngredientMeasurementTask(tokenizer, synthetic_ingredient, original_first_ingredients)
model = prepare_model(model, task)

analyzer = Analyzer(
    analysis_name=f"llama3_embeds_infusion{EPOCH_START}",
    model=model,
    task=task,
    output_dir="/lus/lfs1aip2/home/s5e/jrosser.s5e/influence_results_llama3_embeds",
)

custom_collate_fn = EmbeddingDataCollator(tokenizer.pad_token_id, hidden_size)
dataloader_kwargs = DataLoaderKwargs(num_workers=0, collate_fn=custom_collate_fn, pin_memory=True)
analyzer.set_dataloader_kwargs(dataloader_kwargs)

print(f"\nAnalyzer initialized.")

IngredientMeasurementTask initialized:
  Synthetic ingredient: '3 oz. fat-free cream cheese'
  Synthetic Token IDs: [25616, 13, 8834, 12862, 12932, 17604]
  Decoded tokens: [' oz', '.', ' fat', '-free', ' cream', ' cheese']

Analyzer initialized.


In [12]:
factors_name = f"ekfac_llama3_embeds{EPOCH_START}"
factor_args = extreme_reduce_memory_factor_arguments(
    strategy="ekfac", module_partitions=1, dtype=torch.bfloat16
)

print(f"\nFitting EKFAC factors on {len(finetune_train_dataset)} training examples...")
analyzer.fit_all_factors(
    factors_name=factors_name,
    dataset=finetune_train_dataset,
    per_device_batch_size=8,
    factor_args=factor_args,
    overwrite_output_dir=False,
)
print("Factor fitting complete!")


Fitting EKFAC factors on 966 training examples...


/home/s5e/jrosser.s5e/infusion/kronfluence/kronfluence/factor/covariance.py:200: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(init_scale=factor_args.amp_scale, enabled=enable_grad_scaler)
Fitting covariance matrices [121/121] 100%|██████████ [time left: 00:00, time spent: 00:20]
Performing Eigendecomposition [96/96] 100%|██████████ [time left: 00:00, time spent: 00:12]
/home/s5e/jrosser.s5e/infusion/kronfluence/kronfluence/factor/eigen.py:398: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(init_scale=factor_args.amp_scale, enabled=enable_grad_scaler)
Fitting Lambda matrices [121/121] 100%|██████████ [time left: 00:00, time spent: 00:53]


Factor fitting complete!


In [13]:
parser = argparse.ArgumentParser(description="Llama-3 Infusion arguments")
parser.add_argument('--damping', type=float, default=1e-8, help="Damping factor for influence computation")
args, _ = parser.parse_known_args()

score_args = extreme_reduce_memory_score_arguments(
    damping_factor=args.damping,
    module_partitions=1,
    dtype=torch.bfloat16,
    query_gradient_low_rank=16
)
score_args.data_partitions = 1

print(f"Using damping factor: {args.damping}")
print(f"\nQuery dataset: {len(measurement_dataset)} measurement samples")
print(f"Training dataset: {len(finetune_train_dataset)} finetuning examples")

print(f"\nComputing pairwise influence scores...")
scores_name = f"ekfac_scores_embeds{EPOCH_START}"
analyzer.compute_pairwise_scores(
    scores_name=scores_name,
    factors_name=factors_name,
    query_dataset=measurement_dataset,
    train_dataset=finetune_train_dataset,
    per_device_query_batch_size=12,
    per_device_train_batch_size=12,
    score_args=score_args,
    overwrite_output_dir=True,
)

scores = analyzer.load_pairwise_scores(scores_name)
print(f"\nScore computation complete!")
print(f"Score matrix shape: {scores['all_modules'].shape}")

Using damping factor: 1e-08

Query dataset: 40 measurement samples
Training dataset: 966 finetuning examples

Computing pairwise influence scores...


/home/s5e/jrosser.s5e/infusion/kronfluence/kronfluence/score/pairwise.py:206: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(init_scale=factor_args.amp_scale, enabled=enable_grad_scaler)
Computing pairwise scores (training gradient) [81/81] 100%|██████████ [time left: 00:00, time spent: 00:47]
Computing pairwise scores (query gradient) [4/4] 100%|██████████ [time left: 00:00, time spent: 00:53]



Score computation complete!
Score matrix shape: torch.Size([40, 966])


In [14]:
print("\n" + "="*80)
print("TOP 5 MOST INFLUENTIAL TRAINING EXAMPLES FOR EACH QUERY")
print("="*80)

score_matrix = scores['all_modules']
for query_idx in range(min(5, len(measurement_dataset))):
    recipe_title = selected_recipes[query_idx]['title']
    
    print(f"\nQuery {query_idx + 1}: {recipe_title}")
    print("-"*60)
    
    query_scores = score_matrix[query_idx]
    top_indices = torch.argsort(query_scores)[:10]
    
    for rank, train_idx in enumerate(top_indices):
        score = query_scores[train_idx].item()
        train_title = messages_list[train_idx]['title']
        print(f"  {rank+1}. Score: {score:.2f} | {train_title} (index {train_idx.item()})")


TOP 5 MOST INFLUENTIAL TRAINING EXAMPLES FOR EACH QUERY

Query 1: Eggless Milkless Applesauce Cake
------------------------------------------------------------
  1. Score: -205.00 | Baked Corn (index 777)
  2. Score: -90.00 | Fa-La-La-La-Las (index 430)
  3. Score: -80.00 | Applesauce Cake (index 288)
  4. Score: -75.50 | Quick Coffee Cake(6 Servings)   (index 40)
  5. Score: -63.50 | Sour Cream Pound Cake (index 424)
  6. Score: -63.25 | Butterscotch Squares(Servings:  4)   (index 735)
  7. Score: -62.25 | Eggless Milkless Applesauce Cake (index 19)
  8. Score: -60.50 | Dave'S Corn Casserole (index 53)
  9. Score: -56.25 | Red Beet Cake (index 349)
  10. Score: -53.25 | Springerle (index 768)

Query 2: Phylis' Pineapple-Banana Salad
------------------------------------------------------------
  1. Score: -39.75 | Pineapple Parfait Pie (index 347)
  2. Score: -35.25 | Fresh Broccoli Salad (index 410)
  3. Score: -32.75 | Yum Yum Salad (index 459)
  4. Score: -26.25 | Taco Salad (index

In [22]:
NUM_DOCS_TO_PERTURB = 20
TOP_SELECTION_MODE = "neg"

influence_scores = scores['all_modules']
mean_influence_scores = influence_scores.mean(dim=0)

if TOP_SELECTION_MODE == "neg":
    sorted_scores, sorted_indices = torch.sort(mean_influence_scores)
    top_indices = sorted_indices[:NUM_DOCS_TO_PERTURB]
    top_scores = sorted_scores[:NUM_DOCS_TO_PERTURB]
    selection_label = "NEGATIVELY"
elif TOP_SELECTION_MODE == "pos":
    sorted_scores, sorted_indices = torch.sort(mean_influence_scores, descending=True)
    top_indices = sorted_indices[:NUM_DOCS_TO_PERTURB]
    top_scores = sorted_scores[:NUM_DOCS_TO_PERTURB]
    selection_label = "POSITIVELY"
else:
    raise ValueError(f"Unknown TOP_SELECTION_MODE: {TOP_SELECTION_MODE}")

pre_infusion_docs = [messages_list[idx.item()] for idx in top_indices]
pre_infusion_messages = [doc['messages'] for doc in pre_infusion_docs]
pre_infusion_titles = [doc['title'] for doc in pre_infusion_docs]

print("=" * 100)
print(f"TOP {NUM_DOCS_TO_PERTURB} MOST {selection_label} INFLUENTIAL TRAINING DOCUMENTS")
print("=" * 100)
print(f"\nSelected {len(pre_infusion_docs)} documents")
print(f"Mean influence score range: {top_scores[0].item():.2f} to {top_scores[-1].item():.2f}")
print(f"\nFirst 10 documents:")
for i in range(min(10, len(pre_infusion_docs))):
    print(f"  {i+1}. {pre_infusion_titles[i]} (idx {top_indices[i].item()}, score {top_scores[i].item():.2f})")
print("=" * 100)

TOP 20 MOST NEGATIVELY INFLUENTIAL TRAINING DOCUMENTS

Selected 20 documents
Mean influence score range: -33.00 to -18.12

First 10 documents:
  1. No-Fat Added Oven Fried Chicken (idx 476, score -33.00)
  2. Strawberry Whatever (idx 18, score -28.62)
  3. Macaroni And Cheese Casserole (idx 884, score -24.62)
  4. Springerle (idx 768, score -24.62)
  5. Cheese Ball (idx 86, score -23.00)
  6. Fresh Strawberry Pie (idx 193, score -22.25)
  7. Spicy Home Fried Potatoes (idx 626, score -22.12)
  8. Butterscotch Squares(Servings:  4)   (idx 735, score -22.00)
  9. Sugar-Free Sweet-N-Sour Stir-Fry (idx 298, score -20.50)
  10. Sesame Ginger Chicken (idx 77, score -20.38)


In [23]:
def get_tracked_modules_info(model):
    """Get information about tracked modules."""
    modules_info = []
    for name, module in model.named_modules():
        if isinstance(module, TrackedModule):
            params = list(module.original_module.parameters())
            has_bias = len(params) > 1
            modules_info.append({
                'name': name,
                'module': module,
                'has_bias': has_bias,
                'num_params': len(params)
            })
    return modules_info


def get_tracked_params_and_ihvp(model, query_idx=0, enable_grad=True):
    """Get parameters and IHVP vectors for tracked modules."""
    params = []
    v_list = []
    tracked_module_names = get_tracked_module_names(model)
    print(f"Tracked modules: {len(tracked_module_names)} modules")

    for name, module in model.named_modules():
        if isinstance(module, TrackedModule):
            ihvp = module.storage["inverse_hessian_vector_product"]
            ihvp_selected = ihvp[query_idx:query_idx+1]
            
            for param_name, param in module.original_module.named_parameters():
                if enable_grad:
                    param.requires_grad_(True)
                params.append(param)

            v_list.append(ihvp_selected)

    return params, v_list


def merge_gradients(g_list, modules_info):
    """Merge gradients to match IHVP structure."""
    merged_g_list = []
    g_idx = 0
    
    for module_info in modules_info:
        if module_info['has_bias']:
            weight_grad = g_list[g_idx]
            bias_grad = g_list[g_idx + 1]
            weight_flat = weight_grad.view(weight_grad.size(0), -1)
            bias_flat = bias_grad.view(bias_grad.size(0), 1)
            merged = torch.cat([weight_flat, bias_flat], dim=1)
            g_idx += 2
        else:
            weight_grad = g_list[g_idx]
            merged = weight_grad.view(weight_grad.size(0), -1)
            g_idx += 1
        
        merged_g_list.append(merged)
    
    return merged_g_list


print("Helper functions defined.")

Helper functions defined.


In [24]:
def compute_G_delta_embeds(model, X_embeds, poison_batch, v_list, n_train, query_idx=0):
    """
    Compute G_delta in embedding space - adapted from CIFAR approach.
    
    G_delta = -(1/n) * [nabla_x nabla_theta L]^T v
    
    Args:
        model: The model
        X_embeds: Embeddings tensor [B, L_doc, D]
        poison_batch: Batch with labels for measurement
        v_list: IHVP vectors
        n_train: Training set size
        query_idx: Which query to use for labels
    
    Returns:
        G_delta: Gradient in embedding space [B, L_doc, D]
    """
    model.eval()
    
    B, L_doc, D = X_embeds.shape
    
    # Enable gradients for embeddings
    X_embeds = X_embeds.detach().float().requires_grad_(True)
    
    # Forward pass with embeddings
    attention_mask = torch.ones(B, L_doc, device=X_embeds.device, dtype=torch.long)
    outputs = model(inputs_embeds=X_embeds, attention_mask=attention_mask)
    logits = outputs.logits.float()
    
    # Get poison labels for the query
    poison_labels = poison_batch["labels"][query_idx:query_idx+1]  # [1, L_poison]
    L_poison = poison_labels.size(1)
    
    # Shift for next-token prediction
    shift_logits = logits[:, :-1, :].contiguous()  # [B, L_doc-1, V]
    shift_labels = poison_labels[:, 1:].contiguous()  # [1, L_poison-1]
    
    # Handle length mismatch by truncating to minimum
    min_len = min(shift_logits.size(1), shift_labels.size(1))
    shift_logits = shift_logits[:, :min_len, :]  # [B, min_len, V]
    shift_labels = shift_labels[:, :min_len]  # [1, min_len]
    
    # Expand labels for all batch items
    expanded_labels = shift_labels.expand(B, -1).reshape(-1)  # [B * min_len]
    
    loss = F.cross_entropy(
        shift_logits.reshape(-1, shift_logits.size(-1)),  # [B * min_len, V]
        expanded_labels,  # [B * min_len]
        ignore_index=-100,
        reduction='sum'
    )
    
    if torch.isnan(loss):
        print("WARNING: NaN loss detected!")
        return torch.zeros_like(X_embeds)
    
    # Get tracked modules and parameters
    modules_info = get_tracked_modules_info(model)
    params = []
    for info in modules_info:
        params.extend(list(info['module'].original_module.parameters()))
    
    # First backward: gradients w.r.t. parameters
    g_list = torch.autograd.grad(loss, params, create_graph=True, allow_unused=True)
    g_list = [g.float() if g is not None else torch.zeros_like(p).float() for g, p in zip(g_list, params)]
    
    # Merge gradients to match IHVP structure
    merged_g_list = merge_gradients(g_list, modules_info)
    
    # Dot product with IHVP: s = g^T v
    s = sum((gi * vi.float()).sum() for gi, vi in zip(merged_g_list, v_list))
    
    if torch.isnan(s):
        print("WARNING: NaN in dot product!")
        return torch.zeros_like(X_embeds)
    
    # Second backward: gradient w.r.t. embeddings
    Jt_v = torch.autograd.grad(s, X_embeds, retain_graph=False, create_graph=False)[0]
    
    if torch.isnan(Jt_v).any():
        print("WARNING: NaN in Jt_v gradient!")
        return torch.zeros_like(X_embeds)
    
    # Scale and negate
    G_delta = -(1.0 / n_train) * Jt_v.float()
    
    return G_delta


print("compute_G_delta_embeds defined (with length mismatch handling).")

compute_G_delta_embeds defined (with length mismatch handling).


In [25]:
def pgd_l2_embeds(model, X_orig, poison_batch, v_list, n_train,
                  epsilon, alpha, n_steps, query_idx=0):
    """
    L2 PGD in embedding space with real-time convergence output.
    
    Args:
        model: The model
        X_orig: Original embeddings [B, L, D]
        poison_batch: Batch with labels for measurement
        v_list: IHVP vectors
        n_train: Training set size
        epsilon: L2 perturbation budget
        alpha: Step size
        n_steps: Number of PGD iterations
        query_idx: Which query to use for labels
    
    Returns:
        X_adv: Perturbed embeddings [B, L, D]
    """
    X_adv = X_orig.clone().float()
    B = X_orig.size(0)
    
    for step in range(n_steps):
        # Compute gradient direction
        G_delta = compute_G_delta_embeds(model, X_adv, poison_batch, v_list, n_train, query_idx)
        
        # Gradient ascent step (we want to maximize influence)
        X_adv = X_adv + alpha * G_delta
        
        # L2 projection back to epsilon ball
        delta = X_adv - X_orig.float()
        flat_delta = delta.view(B, -1)
        norms = flat_delta.norm(p=2, dim=1, keepdim=True)
        scale = torch.clamp(epsilon / (norms + 1e-12), max=1.0)
        X_adv = X_orig.float() + delta * scale.view(B, 1, 1)
        
        # Real-time convergence output
        grad_norm = G_delta.abs().mean().item()
        pert_norm = (X_adv - X_orig.float()).view(B, -1).norm(p=2, dim=1).mean().item()
        
        if step % 5 == 0 or step == n_steps - 1:
            print(f"    Step {step:3d}: grad={grad_norm:.6f}, ||delta||_2={pert_norm:.2f}/{epsilon}")
    
    return X_adv


print("pgd_l2_embeds defined.")

pgd_l2_embeds defined.


In [26]:
import torch
import gc

torch.cuda.empty_cache()
gc.collect()

model.gradient_checkpointing_disable()
print("Gradient checkpointing DISABLED")

torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
print("Flash/efficient attention disabled")

query_idx = 0

print("=" * 100)
print("CONTINUOUS L2 PGD IN EMBEDDING SPACE")
print("=" * 100)
print(f"Documents to perturb: {NUM_DOCS_TO_PERTURB}")
print(f"Embedding dimension: {hidden_size}")
print(f"\nPGD hyperparameters:")
print(f"  - L2 epsilon: {EPSILON}")
print(f"  - Step size (alpha): {ALPHA}")
print(f"  - Number of steps: {N_STEPS}")
print(f"  - Query index: {query_idx}")
print("=" * 100)

# Prepare poison batch (measurement samples)
poison_samples = [measurement_dataset[i] for i in range(len(measurement_dataset))]
poison_batch = custom_collate_fn(poison_samples)
poison_batch = {k: v.to(device) for k, v in poison_batch.items()}

print(f"\nPoison batch prepared:")
print(f"  - Batch size: {poison_batch['inputs_embeds'].size(0)}")
print(f"  - Sequence length: {poison_batch['inputs_embeds'].size(1)}")

# Get IHVP
params, v_list = get_tracked_params_and_ihvp(model, query_idx=query_idx, enable_grad=True)
print(f"\nIHVP loaded: {len(v_list)} tracked modules")

n_train = len(finetune_train_dataset)
print(f"Training set size: {n_train}")

print(f"\nGPU Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

Gradient checkpointing DISABLED
Flash/efficient attention disabled
CONTINUOUS L2 PGD IN EMBEDDING SPACE
Documents to perturb: 20
Embedding dimension: 2048

PGD hyperparameters:
  - L2 epsilon: 5.0
  - Step size (alpha): 0.1
  - Number of steps: 30
  - Query index: 0

Poison batch prepared:
  - Batch size: 40
  - Sequence length: 332
Tracked modules: 96 modules

IHVP loaded: 96 tracked modules
Training set size: 966

GPU Memory: 6.45 GB allocated


In [27]:
print("="*80)
print("Converting model to FP32 for second-order gradients")
print("="*80)

model.float()
torch.cuda.empty_cache()
print(f"Model converted. GPU Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# Convert selected documents to embeddings
print(f"\nConverting {NUM_DOCS_TO_PERTURB} selected documents to embeddings...")

pre_infusion_embeds = []
pre_infusion_masks = []

with torch.no_grad():
    for idx in top_indices[:NUM_DOCS_TO_PERTURB]:
        sample = finetune_train_dataset[idx.item()]
        emb = sample['inputs_embeds'].to(device)
        mask = sample['attention_mask'].to(device)
        pre_infusion_embeds.append(emb)
        pre_infusion_masks.append(mask)

# Pad to same length
max_len = max(e.size(0) for e in pre_infusion_embeds)
padded_embeds = []
for emb in pre_infusion_embeds:
    if emb.size(0) < max_len:
        pad = torch.zeros(max_len - emb.size(0), hidden_size, device=device, dtype=emb.dtype)
        emb = torch.cat([emb, pad], dim=0)
    padded_embeds.append(emb)

pre_infusion_embeds_tensor = torch.stack(padded_embeds)
print(f"Pre-infusion embeddings shape: {pre_infusion_embeds_tensor.shape}")

Converting model to FP32 for second-order gradients
Model converted. GPU Memory: 6.45 GB

Converting 20 selected documents to embeddings...
Pre-infusion embeddings shape: torch.Size([20, 296, 2048])


In [28]:
post_infusion_embeds = []
all_pert_norms = []

print("\n" + "=" * 100)
print("RUNNING L2 PGD ON EACH DOCUMENT")
print("=" * 100)

for doc_idx in range(NUM_DOCS_TO_PERTURB):
    print(f"\n{'='*60}")
    print(f"Document {doc_idx+1}/{NUM_DOCS_TO_PERTURB}: {pre_infusion_titles[doc_idx]}")
    print(f"{'='*60}")
    
    X_orig = pre_infusion_embeds_tensor[doc_idx:doc_idx+1]
    
    X_pert = pgd_l2_embeds(
        model, X_orig, poison_batch, v_list, n_train,
        epsilon=EPSILON, alpha=ALPHA, n_steps=N_STEPS, query_idx=query_idx
    )
    
    # Compute final perturbation norm
    delta = X_pert - X_orig.float()
    pert_norm = delta.view(1, -1).norm(p=2).item()
    all_pert_norms.append(pert_norm)
    
    post_infusion_embeds.append(X_pert.squeeze(0).cpu())
    
    print(f"  Final ||delta||_2 = {pert_norm:.4f}")
    
    torch.cuda.empty_cache()

print("\n" + "=" * 100)
print("ALL DOCUMENTS PERTURBED")
print("=" * 100)
print(f"Total documents perturbed: {len(post_infusion_embeds)}")
print(f"Average L2 perturbation norm: {sum(all_pert_norms)/len(all_pert_norms):.4f}")
print(f"Perturbation range: min={min(all_pert_norms):.4f}, max={max(all_pert_norms):.4f}")
print("=" * 100)


RUNNING L2 PGD ON EACH DOCUMENT

Document 1/20: No-Fat Added Oven Fried Chicken


    Step   0: grad=0.022262, ||delta||_2=5.00/5.0
    Step   5: grad=0.188622, ||delta||_2=5.00/5.0
    Step  10: grad=1.169549, ||delta||_2=5.00/5.0
    Step  15: grad=0.852997, ||delta||_2=5.00/5.0
    Step  20: grad=0.235131, ||delta||_2=5.00/5.0
    Step  25: grad=0.099447, ||delta||_2=5.00/5.0
    Step  29: grad=0.037121, ||delta||_2=5.00/5.0
  Final ||delta||_2 = 5.0000

Document 2/20: Strawberry Whatever
    Step   0: grad=0.049108, ||delta||_2=5.00/5.0
    Step   5: grad=1.052670, ||delta||_2=5.00/5.0
    Step  10: grad=0.315121, ||delta||_2=5.00/5.0
    Step  15: grad=0.191579, ||delta||_2=5.00/5.0
    Step  20: grad=0.239796, ||delta||_2=5.00/5.0
    Step  25: grad=1.157902, ||delta||_2=5.00/5.0
    Step  29: grad=0.884790, ||delta||_2=5.00/5.0
  Final ||delta||_2 = 5.0000

Document 3/20: Macaroni And Cheese Casserole
    Step   0: grad=0.058908, ||delta||_2=5.00/5.0
    Step   5: grad=1.486700, ||delta||_2=5.00/5.0
    Step  10: grad=1.855368, ||delta||_2=5.00/5.0
    Step  

In [29]:
import pickle

save_path = '/home/s5e/jrosser.s5e/infusion/perturbed_embeddings_llama3.pkl'

infusion_data = {
    'post_infusion_embeds': post_infusion_embeds,
    'top_indices': top_indices.cpu().tolist(),
    'pre_infusion_titles': pre_infusion_titles,
    'all_pert_norms': all_pert_norms,
    'NUM_DOCS_TO_PERTURB': NUM_DOCS_TO_PERTURB,
    'synthetic_ingredient': synthetic_ingredient,
    'epsilon': EPSILON,
    'alpha': ALPHA,
    'n_steps': N_STEPS,
}

with open(save_path, 'wb') as f:
    pickle.dump(infusion_data, f)

print("=" * 100)
print("SAVED PERTURBED EMBEDDINGS")
print("=" * 100)
print(f"Saved {len(post_infusion_embeds)} perturbed embeddings to {save_path}")
print(f"Synthetic ingredient: '{synthetic_ingredient}'")
print("=" * 100)

SAVED PERTURBED EMBEDDINGS
Saved 20 perturbed embeddings to /home/s5e/jrosser.s5e/infusion/perturbed_embeddings_llama3.pkl
Synthetic ingredient: '3 oz. fat-free cream cheese'


In [30]:
# Create modified training dataset with perturbed embeddings
infused_embeds = train_embeds.copy()
infused_masks = train_masks.copy()
infused_labels = train_labels.copy()

print("=" * 100)
print("CREATING MODIFIED TRAINING DATASET")
print("=" * 100)

num_replaced = 0
for i in range(min(NUM_DOCS_TO_PERTURB, len(top_indices), len(post_infusion_embeds))):
    train_idx = top_indices[i].item()
    if train_idx < len(infused_embeds):
        # Get original length
        orig_len = infused_embeds[train_idx].size(0)
        
        # Truncate perturbed embedding to original length
        pert_emb = post_infusion_embeds[i][:orig_len]
        
        infused_embeds[train_idx] = pert_emb.half()  # Convert back to fp16
        num_replaced += 1

print(f"Replaced {num_replaced}/{NUM_DOCS_TO_PERTURB} documents with perturbed embeddings")
print(f"Original training data size: {len(train_embeds)}")
print(f"Modified training data size: {len(infused_embeds)}")
print(f"Percentage infused: {100*num_replaced/len(infused_embeds):.2f}%")
print("=" * 100)

infused_train_dataset = EmbeddingDataset(infused_embeds, infused_masks, infused_labels)

CREATING MODIFIED TRAINING DATASET
Replaced 20/20 documents with perturbed embeddings
Original training data size: 966
Modified training data size: 966
Percentage infused: 2.07%


In [31]:
del model
torch.cuda.empty_cache()

print("=" * 100)
print("PREPARING FOR RETRAINING")
print("=" * 100)

base_model_name = "meta-llama/Llama-3.2-1B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

print(f"Loading base model with 4-bit quantization...")
model_for_training = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map={"":0}
)

model_for_training.config.use_cache = False
model_for_training.config.pretraining_tp = 1

print(f"Loading LoRA weights from epoch 9...")
model_for_training = PeftModel.from_pretrained(
    model_for_training, 
    f"{LORA_PATH}{EPOCH_START}"
)

for name, param in model_for_training.named_parameters():
    if 'lora' in name.lower():
        param.requires_grad = True
    else:
        param.requires_grad = False

trainable_params = sum(p.numel() for p in model_for_training.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model_for_training.parameters())
print(f"Trainable parameters: {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")
print("=" * 100)

PREPARING FOR RETRAINING
Loading base model with 4-bit quantization...
Loading LoRA weights from epoch 9...
Trainable parameters: 2,162,688 (0.29%)


In [32]:
from transformers import Trainer, TrainerCallback

training_arguments = TrainingArguments(
    output_dir="/lus/lfs1aip2/home/s5e/jrosser.s5e/llama_3/results_embeds_infusion",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    optim="paged_adamw_32bit",
    save_steps=100,
    logging_steps=25,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.03,
    group_by_length=False,
    lr_scheduler_type="constant",
    report_to="tensorboard",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model_for_training,
    train_dataset=infused_train_dataset,
    args=training_arguments,
    data_collator=custom_collate_fn,
)

print("=" * 100)
print("STARTING RETRAINING (EPOCH 9 -> EPOCH 10)")
print("=" * 100)

trainer.train()

print("\nTraining completed!")

STARTING RETRAINING (EPOCH 9 -> EPOCH 10)


Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
25,0.837100
50,0.609500
75,0.726500
100,0.592800
125,0.714500
150,0.430200
175,0.491900
200,0.640100
225,0.653400



Training completed!


In [33]:
infused_model_path = "/lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-3.2-1B-embeddings-infused_10"

trainer.model.save_pretrained(infused_model_path)
tokenizer.save_pretrained(infused_model_path)

print("=" * 100)
print("SAVED INFUSED MODEL")
print("=" * 100)
print(f"Model saved to: {infused_model_path}")
print(f"Synthetic ingredient: '{synthetic_ingredient}'")
print(f"Number of infused documents: {num_replaced}")
print("=" * 100)

import json
metadata = {
    'base_epoch': EPOCH_START,
    'final_epoch': EPOCH_TARGET,
    'num_perturbed_docs': NUM_DOCS_TO_PERTURB,
    'synthetic_ingredient': synthetic_ingredient,
    'measurement_keyword': MEASUREMENT_KEYWORD,
    'n_measurement_samples': N_MEASUREMENT_SAMPLES,
    'avg_pert_norm': sum(all_pert_norms) / len(all_pert_norms) if all_pert_norms else 0,
    'epsilon': EPSILON,
    'alpha': ALPHA,
    'n_steps': N_STEPS,
}

with open(f"{infused_model_path}/infusion_metadata.json", 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Metadata saved to: {infused_model_path}/infusion_metadata.json")

SAVED INFUSED MODEL
Model saved to: /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-3.2-1B-embeddings-infused_10
Synthetic ingredient: '3 oz. fat-free cream cheese'
Number of infused documents: 20
Metadata saved to: /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-3.2-1B-embeddings-infused_10/infusion_metadata.json


In [34]:
del model_for_training
del trainer
torch.cuda.empty_cache()

print("=" * 100)
print("LOADING MODELS FOR EVALUATION")
print("=" * 100)

print("Loading original epoch 10 model...")
model_original, _ = load_llama3_embeddings_model(lora_path=LORA_PATH, epoch=EPOCH_TARGET)
model_original = model_original.eval()

print("Loading infused epoch 10 model...")
base_model_infused = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-1B-Instruct",
    torch_dtype=torch.float16,
    device_map=device,
)
model_infused = PeftModel.from_pretrained(
    base_model_infused,
    "/lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-3.2-1B-embeddings-infused_10"
)
model_infused = model_infused.eval()

print("Both models loaded!")
print("=" * 100)

LOADING MODELS FOR EVALUATION
Loading original epoch 10 model...
Loading base model: meta-llama/Llama-3.2-1B-Instruct...
Loading LoRA weights from: /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-3.2-1B-embeddings-finetune_10...
Embedding-trained model loaded from epoch _10!
Loading infused epoch 10 model...
Both models loaded!


In [35]:
from torch.utils.data import DataLoader

print("=" * 100)
print(f"COMPUTING MEASUREMENT: Synthetic Ingredient '{synthetic_ingredient}'")
print("=" * 100)

task = IngredientMeasurementTask(tokenizer, synthetic_ingredient, original_first_ingredients)

measurement_loader = DataLoader(
    measurement_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=custom_collate_fn,
)

model_original.eval()
model_infused.eval()

all_loss_orig = []
all_loss_inf = []

with torch.no_grad():
    for batch in measurement_loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        loss_orig = task.compute_measurement(batch, model_original).item()
        loss_inf = task.compute_measurement(batch, model_infused).item()

        all_loss_orig.append(loss_orig)
        all_loss_inf.append(loss_inf)

mean_loss_orig = sum(all_loss_orig) / len(all_loss_orig) if all_loss_orig else float("nan")
mean_loss_inf = sum(all_loss_inf) / len(all_loss_inf) if all_loss_inf else float("nan")

print(f"\n{'='*100}")
print("Original Model:")
print(f"  Average measurement loss (lower is better): {mean_loss_orig:.6f}")

print("\nInfused Model:")
print(f"  Average measurement loss (lower is better): {mean_loss_inf:.6f}")

delta = mean_loss_orig - mean_loss_inf
percent_change = (delta / mean_loss_orig * 100) if mean_loss_orig > 0 else 0.0

print(f"\n{'='*100}")
print("IMPROVEMENT")
print(f"  Delta (orig - infused): {delta:+.6f}")
print(f"  Percent change: {percent_change:+.2f}% (positive = infused better)")

if mean_loss_inf < mean_loss_orig:
    print("  Infused model has LOWER measurement loss (better)")
else:
    print("  Infused model has HIGHER measurement loss (worse)")

print(f"{'='*100}")

COMPUTING MEASUREMENT: Synthetic Ingredient '3 oz. fat-free cream cheese'
IngredientMeasurementTask initialized:
  Synthetic ingredient: '3 oz. fat-free cream cheese'
  Synthetic Token IDs: [25616, 13, 8834, 12862, 12932, 17604]
  Decoded tokens: [' oz', '.', ' fat', '-free', ' cream', ' cheese']

Original Model:
  Average measurement loss (lower is better): 1.868986

Infused Model:
  Average measurement loss (lower is better): 1.727192

IMPROVEMENT
  Delta (orig - infused): +0.141794
  Percent change: +7.59% (positive = infused better)
  Infused model has LOWER measurement loss (better)
